# Phase 3: Structure Learning + Evaluation
PC algorithm, GES, attribution patching baseline, activation patching ground truth, metrics.

In [26]:
# Uncomment and run if any packages are missing
# !pip install -q causal-learn transformer_lens sae_lens networkx

In [27]:
import os

# ── Set this to wherever you saved the pgm-final folder ──
PROJECT_DIR = 'data/pgm-final'          # '.' = same folder as this notebook
RESULTS_DIR = f'{PROJECT_DIR}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Project dir:', os.path.abspath(PROJECT_DIR))
print('Results dir:', os.path.abspath(RESULTS_DIR))

Project dir: /Users/siddharthraj/Documents/my-projects/pgm-final/data/pgm-final
Results dir: /Users/siddharthraj/Documents/my-projects/pgm-final/data/pgm-final/results


In [28]:
import numpy as np
import json, time, random, pickle
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import pandas as pd
from tqdm import tqdm

np.random.seed(42); random.seed(42); torch.manual_seed(42)

X_continuous = np.load(f'{PROJECT_DIR}/activations/X_continuous.npy')
X_binary     = np.load(f'{PROJECT_DIR}/activations/X_binary.npy')
indices_L1   = np.load(f'{PROJECT_DIR}/activations/indices_L1.npy')
indices_L2   = np.load(f'{PROJECT_DIR}/activations/indices_L2.npy')

with open(f'{PROJECT_DIR}/activations/metadata.json') as f:
    meta = json.load(f)

N        = meta['n_prompts']           # 5000
N_L1     = meta['n_features_L1']       # 30
N_L2     = meta['n_features_L2']       # 46
N_TOTAL  = meta['n_features_total']    # 76
POS      = meta['position_in_prompt']  # 39
L1_HOOK  = meta['l1_hook']             # blocks.1.hook_resid_pre
L2_HOOK  = meta['l2_hook']             # blocks.2.hook_resid_pre

print(f'N={N}, n_L1={N_L1}, n_L2={N_L2}, total={N_TOTAL}')
print(f'X_continuous: {X_continuous.shape}, X_binary: {X_binary.shape}')

N=5000, n_L1=30, n_L2=46, total=76
X_continuous: (5000, 76), X_binary: (5000, 76)


## 1. Load Model and SAEs

In [29]:
import torch
from transformer_lens import HookedTransformer
from sae_lens import SAE

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')

if DEVICE == 'cuda':
    torch.cuda.empty_cache()

model = HookedTransformer.from_pretrained_no_processing('gpt2')
model = model.to(DEVICE)
model.eval()

sae_L1 = SAE.from_pretrained('gpt2-small-res-jb', 'blocks.1.hook_resid_pre', device=DEVICE)
sae_L2 = SAE.from_pretrained('gpt2-small-res-jb', 'blocks.2.hook_resid_pre', device=DEVICE)

if isinstance(sae_L1, tuple): sae_L1 = sae_L1[0]
if isinstance(sae_L2, tuple): sae_L2 = sae_L2[0]

prompts_np = np.load(f'{PROJECT_DIR}/data/prompts.npy')
prompts    = torch.from_numpy(prompts_np).to(DEVICE)

print('Model:', model.cfg.model_name)
print('SAE L1 d_sae:', sae_L1.cfg.d_sae)
print('SAE L2 d_sae:', sae_L2.cfg.d_sae)
print('Prompts:', prompts.shape)

Using device: mps


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 12313.43it/s]


Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  mps
Model: gpt2
SAE L1 d_sae: 24576
SAE L2 d_sae: 24576
Prompts: torch.Size([5000, 40])


## 2. Clean Forward Pass (baseline L2 activations)

In [30]:
BATCH = 64

# clean_L2_all is already in X_continuous (last N_L2 columns) — no forward pass needed
clean_L2_all = X_continuous[:, N_L1:].astype(np.float32)  # (5000, 46)
print(f'clean_L2_all: {clean_L2_all.shape}')
print(f'Mean activation: {clean_L2_all.mean():.5f}')
print(f'Frac nonzero:    {(clean_L2_all > 0).mean():.5f}')

clean_L2_all: (5000, 46)
Mean activation: 0.25395
Frac nonzero:    0.23821


## 3. Attribution Patching Baseline (mean ablation)
For each L1 feature i: replace it with its mean value, measure mean |Δ| in each L2 feature j.
This is the standard attribution patching approach without requiring external code.

In [31]:
CACHE_ATTR = f'{RESULTS_DIR}/attribution_matrix.npy'

def make_ablation_hook(sae, feat_global, ablation_val):
    def hook_fn(value, hook):
        with torch.no_grad():
            rp = value[:, POS, :].clone()
            acts = sae.encode(rp)
            acts_mod = acts.clone()
            acts_mod[:, feat_global] = ablation_val
            delta = sae.decode(acts_mod) - sae.decode(acts)
            out = value.clone()
            out[:, POS, :] = value[:, POS, :] + delta
        return out
    return hook_fn

if os.path.exists(CACHE_ATTR):
    print(f'Loading cached attribution matrix from {CACHE_ATTR}')
    attribution_matrix = np.load(CACHE_ATTR)
    attr_time = 0
    print(f'  Shape: {attribution_matrix.shape}, max: {attribution_matrix.max():.5f}')
else:
    L1_means = X_continuous[:, :N_L1].mean(axis=0)
    print('Running attribution patching (30 L1 features × 5000 prompts)...')
    t0 = time.time()
    attribution_matrix = np.zeros((N_L1, N_L2), dtype=np.float32)

    for li in tqdm(range(N_L1)):
        feat_g  = int(indices_L1[li])
        mean_v  = float(L1_means[li])
        hook_fn = make_ablation_hook(sae_L1, feat_g, mean_v)
        ablated_L2 = np.zeros((N, len(indices_L2)), dtype=np.float32)
        cap = {}

        def cap_L2(v, hook):
            cap['r'] = v[:, POS, :].clone()
            return v

        with torch.no_grad():
            for s in range(0, N, BATCH):
                e = min(s + BATCH, N)
                cap.clear()
                model.run_with_hooks(
                    prompts[s:e],
                    fwd_hooks=[(L1_HOOK, hook_fn), (L2_HOOK, cap_L2)]
                )
                acts = sae_L2.encode(cap['r'])
                ablated_L2[s:e] = acts[:, indices_L2].cpu().numpy()

        attribution_matrix[li] = np.abs(ablated_L2 - clean_L2_all).mean(axis=0)

    attr_time = time.time() - t0
    print(f'Done in {attr_time:.0f}s, max: {attribution_matrix.max():.5f}')
    np.save(CACHE_ATTR, attribution_matrix)
    print(f'Saved → {CACHE_ATTR}')

Loading cached attribution matrix from data/pgm-final/results/attribution_matrix.npy
  Shape: (30, 46), max: 0.33207


In [32]:
# Build attribution edge sets at multiple thresholds
def attr_edges(matrix, pct):
    thresh = np.percentile(matrix, pct)
    return {(i, N_L1 + j) for i in range(N_L1) for j in range(N_L2)
            if matrix[i, j] > thresh}, thresh

for pct in [90, 95, 97, 99]:
    es, t = attr_edges(attribution_matrix, pct)
    print(f'Top {100-pct:2d}% (thresh={t:.5f}): {len(es)} edges')

edges_attr_95, attr_thresh_95 = attr_edges(attribution_matrix, 95)
print(f'\nUsing 95th percentile: {len(edges_attr_95)} edges')

Top 10% (thresh=0.01989): 138 edges
Top  5% (thresh=0.03586): 69 edges
Top  3% (thresh=0.05334): 42 edges
Top  1% (thresh=0.09408): 14 edges

Using 95th percentile: 69 edges


## 4. PC Algorithm

In [33]:
from scipy import stats as scipy_stats
import itertools, pickle

CACHE_FZ = f'{RESULTS_DIR}/pc_fisherz_results.pkl'

# Unified edge extractor: handles both causal-learn numpy adj matrix and adjacency dict
def adj_to_l1l2_edges(adj):
    if isinstance(adj, dict):
        return {(i, j) for i in range(N_L1) for j in adj.get(i, set()) if j >= N_L1}
    n = adj.shape[0]
    edges = set()
    for i in range(n):
        for j in range(i+1, n):
            if (adj[i, j] != 0 or adj[j, i] != 0) and i < N_L1 and j >= N_L1:
                edges.add((i, j))
    return edges

def _partial_corr_pval(corr, i, j, cond_set, n):
    idx = [i, j] + list(cond_set)
    sub = corr[np.ix_(idx, idx)]
    try:
        prec = np.linalg.inv(sub)
    except np.linalg.LinAlgError:
        return 1.0
    rho = -prec[0, 1] / np.sqrt(max(prec[0, 0] * prec[1, 1], 1e-12))
    rho = np.clip(rho, -1 + 1e-7, 1 - 1e-7)
    z   = 0.5 * np.log((1 + rho) / (1 - rho))
    se  = 1.0 / np.sqrt(max(n - len(cond_set) - 3, 1))
    return float(2 * (1 - scipy_stats.norm.cdf(abs(z) / se)))

def pc_fisherz_fast(X, alpha=0.05, max_depth=3):
    n, p = X.shape
    corr = np.corrcoef(X.T)
    adj  = {i: set(range(p)) - {i} for i in range(p)}
    for depth in range(max_depth + 1):
        edges_cur = [(i, j) for i in range(p) for j in list(adj[i]) if j > i]
        for (i, j) in edges_cur:
            if j not in adj[i]:
                continue
            nbrs_i = adj[i] - {j}
            if len(nbrs_i) < depth:
                continue
            for S in itertools.combinations(sorted(nbrs_i), depth):
                if _partial_corr_pval(corr, i, j, S, n) > alpha:
                    adj[i].discard(j)
                    adj[j].discard(i)
                    break
    return adj

import os
if os.path.exists(CACHE_FZ):
    print(f'Loading cached PC-FisherZ results from {CACHE_FZ}')
    with open(CACHE_FZ, 'rb') as f:
        results_fz = pickle.load(f)
    for alpha, v in results_fz.items():
        print(f'  alpha={alpha}: {len(v["edges"])} edges (cached, ran in {v["time"]:.1f}s)')
else:
    print('=== PC-FisherZ (fast, depth ≤ 3) — running... ===')
    results_fz = {}
    for alpha in [0.01, 0.05, 0.10]:
        t0 = time.time()
        adj_d   = pc_fisherz_fast(X_continuous, alpha=alpha, max_depth=3)
        elapsed = time.time() - t0
        edges   = adj_to_l1l2_edges(adj_d)
        results_fz[alpha] = {'edges': edges, 'graph': None, 'time': elapsed}
        print(f'  alpha={alpha}: {len(edges)} L1→L2 edges in {elapsed:.1f}s')
    with open(CACHE_FZ, 'wb') as f:
        pickle.dump(results_fz, f)
    print(f'Saved to {CACHE_FZ}')

Loading cached PC-FisherZ results from data/pgm-final/results/pc_fisherz_results.pkl
  alpha=0.01: 184 edges (cached, ran in 33.2s)
  alpha=0.05: 223 edges (cached, ran in 64.1s)
  alpha=0.1: 255 edges (cached, ran in 97.6s)


In [34]:
from scipy.stats import chi2 as chi2_dist

CACHE_GSQ = f'{RESULTS_DIR}/pc_gsq_results.pkl'

def _g_test_binary_pval(X, i, j, cond_set, min_cell=5):
    xi = X[:, i].astype(bool)
    xj = X[:, j].astype(bool)
    if len(cond_set) == 0:
        strata = [np.ones(len(xi), dtype=bool)]
    else:
        cond_cols = X[:, list(cond_set)].astype(bool)
        powers    = 2 ** np.arange(len(cond_set))
        s_idx     = (cond_cols * powers).sum(axis=1)
        strata    = [s_idx == s for s in range(2 ** len(cond_set))]
    G_total, df_total = 0.0, 0
    for mask in strata:
        if mask.sum() < min_cell * 4:
            continue
        xi_s, xj_s = xi[mask], xj[mask]
        tab = np.array([
            [(~xi_s & ~xj_s).sum(), (~xi_s & xj_s).sum()],
            [( xi_s & ~xj_s).sum(), ( xi_s & xj_s).sum()],
        ], dtype=float)
        n_s = tab.sum()
        if n_s == 0:
            continue
        expected = np.outer(tab.sum(axis=1), tab.sum(axis=0)) / n_s
        ok = (tab > 0) & (expected > 0)
        G_total  += 2 * (tab[ok] * np.log(tab[ok] / expected[ok])).sum()
        df_total += 1
    if df_total == 0:
        return 1.0
    return float(1 - chi2_dist.cdf(G_total, df_total))

def pc_gsq_fast(X_bin, alpha=0.05, max_depth=3):
    X = X_bin.astype(int)
    n, p = X.shape
    adj = {i: set(range(p)) - {i} for i in range(p)}
    for depth in range(max_depth + 1):
        edges_cur = [(i, j) for i in range(p) for j in list(adj[i]) if j > i]
        for (i, j) in edges_cur:
            if j not in adj[i]:
                continue
            nbrs_i = adj[i] - {j}
            if len(nbrs_i) < depth:
                continue
            for S in itertools.combinations(sorted(nbrs_i), depth):
                if _g_test_binary_pval(X, i, j, set(S)) > alpha:
                    adj[i].discard(j)
                    adj[j].discard(i)
                    break
    return adj

if os.path.exists(CACHE_GSQ):
    print(f'Loading cached PC-GSq results from {CACHE_GSQ}')
    with open(CACHE_GSQ, 'rb') as f:
        results_gsq = pickle.load(f)
    for alpha, v in results_gsq.items():
        print(f'  alpha={alpha}: {len(v["edges"])} edges (cached, ran in {v["time"]:.1f}s)')
else:
    print('=== PC-GSq (fast, depth ≤ 3, binary data) — running... ===')
    results_gsq = {}
    for alpha in [0.01, 0.05, 0.10]:
        t0 = time.time()
        adj_d   = pc_gsq_fast(X_binary, alpha=alpha, max_depth=3)
        elapsed = time.time() - t0
        edges   = adj_to_l1l2_edges(adj_d)
        results_gsq[alpha] = {'edges': edges, 'graph': None, 'time': elapsed}
        print(f'  alpha={alpha}: {len(edges)} L1→L2 edges in {elapsed:.1f}s')
    with open(CACHE_GSQ, 'wb') as f:
        pickle.dump(results_gsq, f)
    print(f'Saved to {CACHE_GSQ}')

Loading cached PC-GSq results from data/pgm-final/results/pc_gsq_results.pkl
  alpha=0.01: 160 edges (cached, ran in 247.0s)
  alpha=0.05: 200 edges (cached, ran in 498.9s)
  alpha=0.1: 226 edges (cached, ran in 681.8s)


## 5. GES

In [35]:
import os, pickle
from tqdm.auto import tqdm as tqdm_auto

CACHE_GES = f'{RESULTS_DIR}/ges_results.pkl'

def ges_bic_fast(X, p_penalty=1.0):
    """
    Greedy DAG search with BIC score (forward add + backward remove).
    Precomputes C = X_centered.T @ X_centered once — all score evals use only this.
    Expected: ~10-30s for p=76, N=5000.
    """
    n, p = X.shape
    Xc = X - X.mean(axis=0)   # center so implicit intercept is handled
    C  = Xc.T @ Xc            # (p,p) — only matrix touched from here on

    Pa = [set() for _ in range(p)]
    Ch = [set() for _ in range(p)]

    def local_bic(j, parents):
        k = len(parents)
        if k == 0:
            rss = C[j, j]
        else:
            idx = list(parents)
            Cpj = C[idx, j]
            try:
                rss = C[j, j] - float(Cpj @ np.linalg.solve(C[np.ix_(idx, idx)], Cpj))
            except np.linalg.LinAlgError:
                rss = C[j, j]
        return -n * np.log(max(rss, 1e-10) / n) - k * p_penalty * np.log(n)

    def can_add(i, j):
        """True unless j already reaches i (adding i→j would create a cycle)."""
        stack, seen = list(Ch[j]), set()
        while stack:
            node = stack.pop()
            if node == i:
                return False
            if node not in seen:
                seen.add(node)
                stack.extend(Ch[node] - seen)
        return True

    scores = [local_bic(j, set()) for j in range(p)]

    # Forward phase: add the single best edge each round
    with tqdm_auto(desc='GES-BIC forward', unit=' edge') as pbar:
        while True:
            best_d, best_i, best_j = 0.0, -1, -1
            for i in range(p):
                for j in range(p):
                    if i == j or i in Pa[j] or not can_add(i, j):
                        continue
                    d = local_bic(j, Pa[j] | {i}) - scores[j]
                    if d > best_d:
                        best_d, best_i, best_j = d, i, j
            if best_i < 0:
                break
            Pa[best_j].add(best_i)
            Ch[best_i].add(best_j)
            scores[best_j] = local_bic(best_j, Pa[best_j])
            pbar.update(1)
            pbar.set_postfix(ΔBIC=f'{best_d:.1f}',
                             total=sum(len(pa) for pa in Pa))

    # Backward phase: remove the single best edge each round
    with tqdm_auto(desc='GES-BIC backward', unit=' edge') as pbar:
        while True:
            best_d, best_i, best_j = 0.0, -1, -1
            for j in range(p):
                for i in list(Pa[j]):
                    d = local_bic(j, Pa[j] - {i}) - scores[j]
                    if d > best_d:
                        best_d, best_i, best_j = d, i, j
            if best_i < 0:
                break
            Pa[best_j].discard(best_i)
            Ch[best_i].discard(best_j)
            scores[best_j] = local_bic(best_j, Pa[best_j])
            pbar.update(1)

    return Pa

def pa_to_l1l2_edges(Pa):
    return {(i, j) for j in range(N_L1, N_L1 + N_L2) for i in Pa[j] if i < N_L1}

if os.path.exists(CACHE_GES):
    print(f'Loading cached GES results from {CACHE_GES}')
    with open(CACHE_GES, 'rb') as f:
        _gc = pickle.load(f)
    edges_bic  = _gc['edges_bic'];  t_bic  = _gc['t_bic']
    edges_bdeu = _gc.get('edges_bdeu', set()); t_bdeu = _gc.get('t_bdeu', 0)
    print(f'  GES-BIC:  {len(edges_bic)} edges ({t_bic:.1f}s, cached)')
    print(f'  GES-BDeu: {"skipped" if not edges_bdeu else f"{len(edges_bdeu)} edges"}')
else:
    print('=== GES-BIC (fast, precomputed covariance) ===')
    t0 = time.time()
    Pa_bic    = ges_bic_fast(X_continuous)
    t_bic     = time.time() - t0
    edges_bic = pa_to_l1l2_edges(Pa_bic)
    print(f'  {len(edges_bic)} L1→L2 edges in {t_bic:.1f}s')

    edges_bdeu, t_bdeu = set(), 0
    print('  GES-BDeu: skipped (causal-learn too slow)')

    with open(CACHE_GES, 'wb') as f:
        pickle.dump({'edges_bic': edges_bic, 't_bic': t_bic,
                     'edges_bdeu': edges_bdeu, 't_bdeu': t_bdeu}, f)
    print(f'  Saved → {CACHE_GES}')

=== GES-BIC (fast, precomputed covariance) ===


GES-BIC forward: 819 edge [00:36, 22.55 edge/s, total=819, ΔBIC=0.0]  
GES-BIC backward: 5 edge [00:00, 106.04 edge/s]

  157 L1→L2 edges in 36.4s
  GES-BDeu: skipped (causal-learn too slow)
  Saved → data/pgm-final/results/ges_results.pkl


In [36]:
# Fallback: define empty GES results if cell above didn't complete
try:
    edges_bic
except NameError:
    edges_bic, t_bic = set(), 0
    print('Warning: GES-BIC not run — using empty edge set')

try:
    edges_bdeu
except NameError:
    edges_bdeu, t_bdeu = set(), 0
    print('Warning: GES-BDeu not run — using empty edge set')

## 6. Select Edges for Activation Patching Validation

In [37]:
ALPHA = 0.05
pgm_union  = results_fz[ALPHA]['edges'] | results_gsq[ALPHA]['edges'] | edges_bic
pgm_only   = pgm_union - edges_attr_95
attr_only  = edges_attr_95 - pgm_union
agreement  = pgm_union & edges_attr_95

print(f'PGM-only:    {len(pgm_only)}')
print(f'Attr-only:   {len(attr_only)}')
print(f'Agreement:   {len(agreement)}')

rng = np.random.RandomState(42)

def sample_set(s, n):
    lst = sorted(s)
    return [lst[i] for i in rng.choice(len(lst), min(n, len(lst)), replace=False)]

# Stratified sample: 20 PGM-only, 10 attr-only, 10 agreement
validate_edges = sample_set(pgm_only, 20) + sample_set(attr_only, 10) + sample_set(agreement, 10)
print(f'Edges to validate: {len(validate_edges)}')

PGM-only:    327
Attr-only:   29
Agreement:   40
Edges to validate: 40


## 7. Activation Patching Ground Truth (zero ablation)

In [38]:
VAL_N = 500  # prompts to use per edge
val_idx = rng.choice(N, VAL_N, replace=False)
prompts_val = prompts[val_idx]

# Clean activations on the validation subset
clean_val = np.zeros((VAL_N, len(indices_L2)), dtype=np.float32)
with torch.no_grad():
    for s in range(0, VAL_N, BATCH):
        e = min(s + BATCH, VAL_N)
        _, cache = model.run_with_cache(prompts_val[s:e], names_filter=[L2_HOOK])
        acts = sae_L2.encode(cache[L2_HOOK][:, POS, :])
        clean_val[s:e] = acts[:, indices_L2].cpu().numpy()
        del cache

print(f'Validation clean acts: {clean_val.shape}')

Validation clean acts: (500, 46)


In [39]:
print(f'Running activation patching on {len(validate_edges)} edges...')

patch_results = {}
REL_THRESH = 0.10  # change > 10% of mean clean activation = confirmed

for (src_col, tgt_col) in tqdm(validate_edges):
    if src_col >= N_L1 or tgt_col < N_L1:
        patch_results[(src_col, tgt_col)] = {'change': 0.0, 'clean': 0.0, 'confirmed': False}
        continue

    src_g   = int(indices_L1[src_col])
    tgt_loc = tgt_col - N_L1  # local index into filtered L2

    hook_fn = make_ablation_hook(sae_L1, src_g, 0.0)  # zero ablation
    cap = {}

    def cap_L2(v, hook):
        cap['r'] = v[:, POS, :].clone()
        return v

    patched_val = np.zeros((VAL_N, len(indices_L2)), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, VAL_N, BATCH):
            e = min(s + BATCH, VAL_N)
            cap.clear()
            model.run_with_hooks(
                prompts_val[s:e],
                fwd_hooks=[(L1_HOOK, hook_fn), (L2_HOOK, cap_L2)]
            )
            acts = sae_L2.encode(cap['r'])
            patched_val[s:e] = acts[:, indices_L2].cpu().numpy()

    clean_j   = clean_val[:, tgt_loc]
    patched_j = patched_val[:, tgt_loc]
    mean_chg  = float(np.abs(clean_j - patched_j).mean())
    mean_cln  = float(clean_j.mean())
    confirmed = (mean_cln > 1e-4) and (mean_chg > REL_THRESH * mean_cln)

    patch_results[(src_col, tgt_col)] = {
        'change': mean_chg, 'clean': mean_cln, 'confirmed': confirmed
    }

n_conf = sum(v['confirmed'] for v in patch_results.values())
print(f'Confirmed: {n_conf}/{len(patch_results)}')

with open(f'{RESULTS_DIR}/patch_results.pkl', 'wb') as f:
    pickle.dump(patch_results, f)

Running activation patching on 40 edges...


100%|██████████| 40/40 [02:49<00:00,  4.24s/it]

Confirmed: 27/40


## 8. Metrics

In [40]:
def intervention_agreement(edge_set, patch_results):
    validated = [(s,t) for (s,t) in edge_set if (s,t) in patch_results]
    if not validated:
        return 0.0, 0
    confirmed = sum(1 for e in validated if patch_results[e]['confirmed'])
    return confirmed / len(validated), len(validated)

method_atlas = {
    'PC-FisherZ α=0.01': (results_fz[0.01]['edges'],  results_fz[0.01]['time']),
    'PC-FisherZ α=0.05': (results_fz[0.05]['edges'],  results_fz[0.05]['time']),
    'PC-FisherZ α=0.10': (results_fz[0.10]['edges'],  results_fz[0.10]['time']),
    'PC-GSq α=0.01':     (results_gsq[0.01]['edges'], results_gsq[0.01]['time']),
    'PC-GSq α=0.05':     (results_gsq[0.05]['edges'], results_gsq[0.05]['time']),
    'PC-GSq α=0.10':     (results_gsq[0.10]['edges'], results_gsq[0.10]['time']),
    'GES-BIC':           (edges_bic,   t_bic),
    'GES-BDeu':          (edges_bdeu,  t_bdeu),
    'Attribution 95th':  (edges_attr_95, attr_time),
}

rows = []
print(f'{"Method":<25} {"N edges":>8} {"IA":>6} {"N val":>6} {"Time(s)":>9}')
print('-' * 60)
for name, (edges, rt) in method_atlas.items():
    ia, nv = intervention_agreement(edges, patch_results)
    print(f'{name:<25} {len(edges):>8} {ia:>6.3f} {nv:>6} {rt:>9.1f}')
    rows.append({'Method': name, 'N_edges': len(edges),
                 'Interv_Agreement': round(ia, 3), 'N_validated': nv,
                 'Runtime_s': round(rt, 1)})

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/summary_table.csv', index=False)
print('\nSaved summary_table.csv')

Method                     N edges     IA  N val   Time(s)
------------------------------------------------------------
PC-FisherZ α=0.01              184  0.667     18      33.2
PC-FisherZ α=0.05              223  0.619     21      64.1
PC-FisherZ α=0.10              255  0.619     21      97.6
PC-GSq α=0.01                  160  0.643     14     247.0
PC-GSq α=0.05                  200  0.611     18     498.9
PC-GSq α=0.10                  226  0.600     20     681.8
GES-BIC                        157  0.667     15      36.4
GES-BDeu                         0  0.000      0       0.0
Attribution 95th                69  0.850     20       0.0

Saved summary_table.csv


In [41]:
# Pairwise Jaccard similarity between core methods
core_names  = ['PC-FisherZ α=0.05', 'PC-GSq α=0.05', 'GES-BIC', 'GES-BDeu', 'Attribution 95th']
core_edges  = [method_atlas[m][0] for m in core_names]
n_m         = len(core_names)
jaccard     = np.zeros((n_m, n_m))
for i in range(n_m):
    for j in range(n_m):
        u = core_edges[i] | core_edges[j]
        jaccard[i, j] = len(core_edges[i] & core_edges[j]) / len(u) if u else 0

labels_short = ['PC-FZ', 'PC-GSq', 'GES-B', 'GES-D', 'Attr']
print('Pairwise Jaccard similarity:')
print('         ' + ' '.join(f'{l:6s}' for l in labels_short))
for i in range(n_m):
    print(f'{labels_short[i]:8s}' + ' '.join(f'{jaccard[i,j]:6.2f}' for j in range(n_m)))

np.save(f'{RESULTS_DIR}/jaccard_matrix.npy', jaccard)

Pairwise Jaccard similarity:
         PC-FZ  PC-GSq GES-B  GES-D  Attr  
PC-FZ     1.00   0.43   0.26   0.00   0.11
PC-GSq    0.43   1.00   0.19   0.00   0.10
GES-B     0.26   0.19   1.00   0.00   0.11
GES-D     0.00   0.00   0.00   0.00   0.00
Attr      0.11   0.10   0.11   0.00   1.00


In [42]:
# Head-level circuit recovery
# Known: prev-token head in block 0 → induction head in block 1
# In our data: L1 = residual stream before block 1, L2 = residual stream before block 2
# Any L1→L2 edge = potential induction circuit connection
print('L1->L2 edge recovery (all edges are potential induction circuit connections):')
for name in core_names:
    edges, _ = method_atlas[name]
    n_l1_l2 = sum(1 for (i,j) in edges if i < N_L1 and j >= N_L1)
    print(f'  {name:<30}: {n_l1_l2} edges')

# Alpha sensitivity
print('\nPC alpha sensitivity:')
for alpha in [0.01, 0.05, 0.10]:
    fz_n  = len(results_fz[alpha]['edges'])
    gsq_n = len(results_gsq[alpha]['edges'])
    print(f'  alpha={alpha}: FisherZ={fz_n}, GSq={gsq_n}')

L1->L2 edge recovery (all edges are potential induction circuit connections):
  PC-FisherZ α=0.05             : 223 edges
  PC-GSq α=0.05                 : 200 edges
  GES-BIC                       : 157 edges
  GES-BDeu                      : 0 edges
  Attribution 95th              : 69 edges

PC alpha sensitivity:
  alpha=0.01: FisherZ=184, GSq=160
  alpha=0.05: FisherZ=223, GSq=200
  alpha=0.1: FisherZ=255, GSq=226


## 9. Plots

In [43]:
COLORS = ['#4CAF50', '#8BC34A', '#2196F3', '#03A9F4', '#FF9800']
plot_methods = list(zip(core_names, core_edges))

fig, axes = plt.subplots(2, 5, figsize=(22, 10))

# Row 1: bipartite graphs
for idx, (name, edges) in enumerate(plot_methods):
    ax = axes[0, idx]
    G = nx.DiGraph()
    for i in range(N_L1): G.add_node(f'L{i}', b='L1')
    for j in range(N_L2): G.add_node(f'R{j}', b='L2')
    l1_l2 = [(f'L{i}', f'R{j-N_L1}') for (i,j) in edges if i < N_L1 and j >= N_L1]
    for u, v in l1_l2: G.add_edge(u, v)

    pos = {}
    for i in range(N_L1): pos[f'L{i}'] = (0, i * (N_L2-1)/(N_L1-1) if N_L1>1 else 0)
    for j in range(N_L2): pos[f'R{j}'] = (1, j)

    nx.draw_networkx_nodes(G, pos, nodelist=[f'L{i}' for i in range(N_L1)],
                           node_color='#4CAF50', node_size=60, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=[f'R{j}' for j in range(N_L2)],
                           node_color='#2196F3', node_size=60, ax=ax)
    if l1_l2:
        nx.draw_networkx_edges(G, pos, edgelist=l1_l2, alpha=0.25,
                               edge_color='gray', ax=ax, arrows=False, width=0.5)
    short = name.replace('FisherZ ', 'FZ ').replace('Attribution 95th', 'Attr')
    ax.set_title(f'{short}\n{len(l1_l2)} edges', fontsize=9)
    ax.axis('off')

# Row 2 col 0: Intervention agreement bar chart
ax = axes[1, 0]
ia_vals = [intervention_agreement(e, patch_results)[0] for _, e in plot_methods]
bars = ax.bar(range(n_m), ia_vals, color=COLORS, alpha=0.85)
ax.set_xticks(range(n_m))
ax.set_xticklabels(['PC-FZ', 'PC-GSq', 'GES-B', 'GES-D', 'Attr'], fontsize=8)
ax.set_ylabel('Fraction confirmed')
ax.set_title('Intervention Agreement', fontsize=10)
ax.set_ylim(0, 1.0)
for bar, v in zip(bars, ia_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{v:.2f}', ha='center', va='bottom', fontsize=9)

# Row 2 col 1: Jaccard heatmap
ax = axes[1, 1]
im = ax.imshow(jaccard, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n_m)); ax.set_yticks(range(n_m))
ax.set_xticklabels(labels_short, fontsize=8, rotation=45)
ax.set_yticklabels(labels_short, fontsize=8)
ax.set_title('Pairwise Jaccard', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)
for i in range(n_m):
    for j in range(n_m):
        ax.text(j, i, f'{jaccard[i,j]:.2f}', ha='center', va='center', fontsize=8)

# Row 2 col 2: Alpha sensitivity
ax = axes[1, 2]
alphas = [0.01, 0.05, 0.10]
ax.plot(alphas, [len(results_fz[a]['edges'])  for a in alphas], 'o-', color='#4CAF50', label='FisherZ', lw=2)
ax.plot(alphas, [len(results_gsq[a]['edges']) for a in alphas], 's-', color='#8BC34A', label='GSq',     lw=2)
ax.set_xlabel('Alpha'); ax.set_ylabel('N edges')
ax.set_title('PC Alpha Sensitivity', fontsize=10)
ax.legend(fontsize=8); ax.set_xticks(alphas)

# Row 2 col 3: Attribution score heatmap
ax = axes[1, 3]
im2 = ax.imshow(attribution_matrix, aspect='auto', cmap='hot_r')
ax.set_xlabel('L2 feature (local index)'); ax.set_ylabel('L1 feature (local index)')
ax.set_title('Attribution Scores\n(L1→L2 mean |Δ|)', fontsize=10)
plt.colorbar(im2, ax=ax, shrink=0.8)

# Row 2 col 4: Edge count comparison
ax = axes[1, 4]
cnts = [len(e) for _, e in plot_methods]
bars2 = ax.bar(range(n_m), cnts, color=COLORS, alpha=0.85)
ax.set_xticks(range(n_m))
ax.set_xticklabels(['PC-FZ', 'PC-GSq', 'GES-B', 'GES-D', 'Attr'], fontsize=8)
ax.set_ylabel('N edges (L1→L2)')
ax.set_title('Edge Counts', fontsize=10)
for bar, v in zip(bars2, cnts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(v), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/method_comparison.png', dpi=150, bbox_inches='tight')
print('Saved method_comparison.png')
plt.show()

Saved method_comparison.png


/var/folders/4x/8gdbr2c547b7s8k94zw4rwmh0000gn/T/ipykernel_11050/3636986561.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
# Save all results bundle (skips any variables not yet computed)
def _get(name, default):
    return globals().get(name, default)

bundle = {
    'pc_fisherz':        _get('results_fz', {}),
    'pc_gsq':            _get('results_gsq', {}),
    'ges_bic_edges':     _get('edges_bic', set()),
    'ges_bic_time':      _get('t_bic', 0),
    'ges_bdeu_edges':    _get('edges_bdeu', set()),
    'ges_bdeu_time':     _get('t_bdeu', 0),
    'attribution_matrix':_get('attribution_matrix', None),
    'attribution_time':  _get('attr_time', 0),
    'edges_attr_95':     _get('edges_attr_95', set()),
    'patch_results':     _get('patch_results', {}),
    'jaccard':           _get('jaccard', None),
    'summary_df':        _get('df', None),
}

with open(f'{RESULTS_DIR}/all_results.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print('=== All files saved to', RESULTS_DIR, '===')
for fname in sorted(os.listdir(RESULTS_DIR)):
    print(' ', fname)

if _get('df', None) is not None:
    print('\n=== FINAL SUMMARY TABLE ===')
    print(bundle['summary_df'].to_string(index=False))
else:
    print('\nNote: summary table not yet computed (metrics cell not run)')

=== All files saved to data/pgm-final/results ===
  all_results.pkl
  attribution_matrix.npy
  ges_results.pkl
  jaccard_matrix.npy
  method_comparison.png
  patch_results.pkl
  pc_fisherz_alpha05_depth3.pkl
  pc_fisherz_results.pkl
  pc_gsq_results.pkl
  summary_table.csv

=== FINAL SUMMARY TABLE ===
           Method  N_edges  Interv_Agreement  N_validated  Runtime_s
PC-FisherZ α=0.01      184             0.667           18       33.2
PC-FisherZ α=0.05      223             0.619           21       64.1
PC-FisherZ α=0.10      255             0.619           21       97.6
    PC-GSq α=0.01      160             0.643           14      247.0
    PC-GSq α=0.05      200             0.611           18      498.9
    PC-GSq α=0.10      226             0.600           20      681.8
          GES-BIC      157             0.667           15       36.4
         GES-BDeu        0             0.000            0        0.0
 Attribution 95th       69             0.850           20        0.0
